# 文学研究のためのメタデータ実習（2026年10月16日）

Open Library・Wikidata・OpenAlexを通じて「広く読まれた」「忘れられた」といった文学史の言説を、メタデータの範囲内で検討します。

## はじめに
1. 上の「ランタイム」→「すべてのセルを実行」、または各セル左の ▶ を押します。Googleアカウントへのログインとランタイム接続が求められる場合があります。
2. 以下の入力欄に作品名や年代を指定できます。Pythonの文法を覚える必要はありません。
3. CSVは公開GitHubから自動的に読み込みます。アップロードは必要ありません。
4. 「OpenAlex候補数」は論文による確定的引用数ではありません。Open LibraryのEdition数も刊行部数や読者数ではありません。

注：このノートは公開前のレビュー用です。公開CSVのURLはGitHubへの配布を承認した後に有効になります。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# 公開用データのブランチ・ファイルを変更するときはここだけ修正します。
REPO = "haruka0221/canon-pipeline"
REF = "workshop-20261016"  # 公開時に確定したブランチまたはコミットを指定
BASE = f"https://raw.githubusercontent.com/{REPO}/{REF}/workshop-20261016/data"
DATASET = "workshop_sample.csv" # @param ["workshop_sample.csv", "ol_wikidata_openalex_v1.csv"]
URL = f"{BASE}/{DATASET}"
try:
    df = pd.read_csv(URL, dtype=str, keep_default_na=False, encoding="utf-8-sig")
except Exception as exc:
    raise RuntimeError("CSVを読み込めません。まだGitHubで未公開の場合、担当者が公開後に再実行してください。URL: " + URL) from exc
print(f"読み込み成功：{len(df):,} 行 / {len(df.columns)} 列")
display(df.head(5))

## 1. 作品のタイトル・著者から検索する
部分一致で検索します。空欄を指定すると、その条件では絞り込みません。

In [ ]:
TITLE = "News from Nowhere" # @param {type:"string"}
AUTHOR = "" # @param {type:"string"}
q = df.copy()
if TITLE.strip(): q = q[q["ol_title"].str.contains(TITLE, case=False, regex=False, na=False)]
if AUTHOR.strip(): q = q[q["ol_author_name"].str.contains(AUTHOR, case=False, regex=False, na=False)]
print(f"{len(q)} 行見つかりました")
display(q[["ol_title","ol_author_name","ol_first_publish_year","ol_edition_count","wikidata_decision","openalex_candidate_pairs","identity_status"]].head(30))

## 2. 年代で作品の集合を見る
指定する年はOpen Libraryのソース上の最初の出版年で、厳密な文学史上の初出年とは限りません。講習用の標本CSVは全作品を網羅しません。

In [ ]:
START_YEAR = 1880 # @param {type:"integer"}
END_YEAR = 1900 # @param {type:"integer"}
years = pd.to_numeric(df["ol_first_publish_year"], errors="coerce")
period = df[years.between(START_YEAR, END_YEAR)].copy()
print(f"この読み込みデータ内で{START_YEAR}–{END_YEAR}年に該当：{len(period):,} OLレコード")
display(period[["ol_title","ol_author_name","ol_first_publish_year"]].head(25))

## 3. Edition数を比較する
この列が入力CSVで欠損の場合は、数値が記録されていないことを示します。Editionは実際の刊行回数・刷数・売上・読者数ではありません。

In [ ]:
PLOT_TOP = 15 # @param {type:"integer", min:5, max:40}
x = period.copy()
x["edition_numeric"] = pd.to_numeric(x["ol_edition_count"], errors="coerce")
x = x.dropna(subset=["edition_numeric"]).nlargest(PLOT_TOP,"edition_numeric")
if x.empty: print("この標本にはEdition数が記録されていません。")
else:
    display(x[["ol_title","ol_author_name","ol_edition_count"]])
    ax=x.sort_values("edition_numeric").plot.barh(x="ol_title",y="edition_numeric",legend=False,figsize=(9,max(4,len(x)*.38)))
    ax.set(xlabel="Open Libraryに登録されたEdition数",ylabel="OL title"); plt.tight_layout(); plt.show()

## 4. Wikidataとの照合状況
MATCHは採用された照合、AMBIGUOUSは曖昧、NO_MATCHは現行の探索・判定手続きでは照合できなかったことを意味します。

In [ ]:
d = df["wikidata_decision"].replace("", "NOT_RECORDED").value_counts(dropna=False)
display(d.to_frame("OL source records"))
d.plot.bar(figsize=(7,3),rot=0,title="Wikidata resolution status");plt.tight_layout();plt.show()

## 5. OpenAlexの候補数を見る
同じProject Wに複数のOpen Library source Workが対応する場合、同じ指標が複数行に現れます。Wで重複除去したうえで比較します。未計測（NA）と0件を混同しません。

In [ ]:
MIN_CANDIDATES = 1 # @param {type:"integer", min:0}
active=df[(df["identity_status"]=="ACTIVE_MAPPED") & (df["project_work_id"]!="")].copy()
unique_works=active.drop_duplicates("project_work_id").copy()
unique_works["candidate_numeric"]=pd.to_numeric(unique_works["openalex_candidate_pairs"],errors="coerce")
shown=unique_works[unique_works["candidate_numeric"].ge(MIN_CANDIDATES)].sort_values("candidate_numeric",ascending=False)
print(f"条件に該当するProject W: {len(shown):,}（読み込みデータ内）")
display(shown[["ol_title","ol_author_name","project_work_id","openalex_candidate_pairs","openalex_measurement_status"]].head(25))
print("未計測など:")
display(df["openalex_measurement_status"].value_counts(dropna=False).to_frame("OL rows"))

## 6. 観察と解釈

- Edition数の多さは、読者が多かったことを意味するでしょうか。出版年・出版社・収録時点の偏りを考えてみましょう。
- WikidataでNO_MATCHだった作品は「存在しない作品」でしょうか。
- OpenAlex候補が多い作品は、学術的評価が高いと断定できるでしょうか。同名作品・一般語タイトル・タイトルと著者の誤結合が起こりえます。
- Open Library Workと研究上のProject Workは同一の概念でしょうか。今回の17件の過剰統合の修復事例を考えてみましょう。

### 発展：一作品のEditionを深く見る
この配布CSVは主にWorkごとのメタデータです。Editionの出版社・刊行年・言語を年表で扱うためには、事前に監査した別のEditionテーブルが必要です。未収集の詳細をこのCSVから補完したように表示しません。